# Assignment 1 — South German Credit
Data Science for Business · 2026–2027

Student 1: Flioris Vassily r1142362
Student 2: name and r-number

Complete this notebook in Python. Keep your code and relevant outputs here; write the report in `A1_Template.docx`. The full instructions are in `A1_Assignment.pdf`.

In Colab: upload this `.ipynb` to Drive, open it with **Google Colaboratory**, then click **Connect**. Upload `South_German_Credit.csv` through **Files**. Locally, keep the CSV in the working folder or update `DATA_PATH`.


## Problem Overview


**Context.** We act as consultants at Leuven Analytics Experts & Partners (LAE). A regional bank
has asked us to explore its credit risk data, build a logistic regression model, and turn it into
an explainable scorecard.

**Dataset.** `South_German_Credit.csv` — 1,000 credit records from 1973–1975.
- 21 columns: 20 predictors + 1 target.
- 700 good credits, 300 bad credits.
- No missing values.

**Predictors.**
- 3 quantitative: `duration` (months), `amount` (after an undocumented monotonic transformation,
  so not interpretable in DM), `age` (years).
- 17 categorical: stored as numeric category codes (status, credit_history, purpose, savings,
  employment_duration, installment_rate, personal_status_sex, other_debtors, present_residence,
  property, other_installment_plans, housing, number_credits, job, people_liable, telephone,
  foreign_worker).

**Target.** `credit_risk` — 0 = bad credit, 1 = good credit.

**Deliverables.**
1. Explore the data and split it 70/30 (stratified).
2. Build a logistic regression using WOE-encoded predictors, evaluate with IV, ROC and AUC.
3. Construct a scorecard (600 points at 50:1 good:bad odds, 20 points to double the odds) and
   verify it against the model probabilities.

## 0. Load the dataset
Run the two cells below, then start Question 1.


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Add other imports as needed for your analysis.

In [3]:
DATA_PATH = Path("data/South_German_Credit.csv")
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        "Upload South_German_Credit.csv to the Colab Files panel, "
        "or set DATA_PATH to the CSV location on your computer."
    )

credit = pd.read_csv(DATA_PATH)
display(credit.head())

,status,duration,credit_history,purpose,amount,savings,employment_duration,installment_rate,personal_status_sex,other_debtors,...,property,age,other_installment_plans,housing,number_credits,job,people_liable,telephone,foreign_worker,credit_risk
0,1,18,4,2,1049,1,2,4,2,1,...,2,21,3,1,1,3,2,1,2,1
1,1,9,4,0,2799,1,3,2,3,1,...,1,36,3,1,2,3,1,1,2,1
2,2,12,2,9,841,2,4,2,2,1,...,1,23,3,1,1,2,2,1,2,1
3,1,12,4,0,2122,1,3,3,3,1,...,1,39,3,1,2,2,1,1,1,1
4,1,12,4,0,2171,1,3,4,3,1,...,2,38,1,2,2,2,2,1,1,1


## 1. Explore the data

Follow Question 1 on page 3: check the data, create the target (1 = bad, 0 = good) and make the required stratified 70/30 split before exploring relationships.

Choose two variables and compare their joint groups in one figure. Put your choices, figure and interpretation in report Part 1. Keep checks and supporting calculations here.

### Question 1 — Explore the data

The bank wants to understand how applicant and loan characteristics relate to bad credit.
The goal here is to explore the data and explain one useful pattern before we touch any model.

#### 1. Load and check the data

Load South_German_Credit.csv into your Python notebook. Check the dimensions, variable
types, missing values, duplicate rows and target counts. Create a modelling target with 1 = bad
and 0 = good; keep the original credit_risk column. Record your checks and any action taken in
the notebook.

In [4]:
# 1. Basic checks on the dataset

print("Shape:", credit.shape)
print("\nData types:\n", credit.dtypes)
print("\nMissing values per column:\n", credit.isna().sum())
print("\nDuplicate rows:", credit.duplicated().sum())

# Target counts (original credit_risk: 0 = bad, 1 = good)
print("\ncredit_risk counts:\n", credit["credit_risk"].value_counts())
print("\ncredit_risk proportions:\n", credit["credit_risk"].value_counts(normalize=True))

# Create modelling target: 1 = bad, 0 = good
credit["bad"] = 1 - credit["credit_risk"]
print("\nbad counts:\n", credit["bad"].value_counts())

Shape: (1000, 21)

Data types:
 status                     int64
duration                   int64
credit_history             int64
purpose                    int64
amount                     int64
savings                    int64
employment_duration        int64
installment_rate           int64
personal_status_sex        int64
other_debtors              int64
present_residence          int64
property                   int64
age                        int64
other_installment_plans    int64
housing                    int64
number_credits             int64
job                        int64
people_liable              int64
telephone                  int64
foreign_worker             int64
credit_risk                int64
dtype: object

Missing values per column:
 status                     0
duration                   0
credit_history             0
purpose                    0
amount                     0
savings                    0
employment_duration        0
installment_rate           0


#### 2. Keep a test sample separate

Before selecting predictors or exploring their relationship with the target, make a reproducible,
stratified 70% training / 30% test split. Record the random seed. Use this split throughout the
assignment: training data for exploration and model development, test data only for final
evaluation.

In [5]:
from sklearn.model_selection import train_test_split

SEED = 42

train, test = train_test_split(
    credit,
    test_size=0.30,
    stratify=credit["bad"],
    random_state=SEED,
)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("\nTrain bad rate:", train["bad"].mean().round(4))
print("Test bad rate:", test["bad"].mean().round(4))

Train shape: (700, 22)
Test shape: (300, 22)

Train bad rate: 0.3
Test bad rate: 0.3


#### 3. Choose two variables and make one figure

Choose two variables and justify your choices. Define groups using the combination of both variables, grouping quantitative values or combining categories where appropriate. Make one clearly labelled figure showing each group's bad-credit rate and number of observations. Use descriptive category labels and explain your grouping choices.

#### Variable typing

Before exploring relationships, we classify the predictors by their measurement type.
This classification is used later to decide how to group (Q1.3) and encode (Q2).

We use four groups:

- **Quantitative**: real measured values.
- **Binary categorical**: exactly two categories.
- **Ordinal categorical**: categories with a natural order.
- **Nominal categorical**: categories with no natural order.

In [6]:
quantitative_cols = [
    "duration",
    "amount",
    "age",
]

binary_categorical_cols = [
    "people_liable",
    "telephone",
    "foreign_worker",
]

ordinal_categorical_cols = [
    "status",                 # checking account balance, 1 none to 4 richest
    "credit_history",         # 0 past delays to 4 all repaid duly
    "savings",                # 1 none/unknown to 5 ≥ 1,000 DM
    "employment_duration",    # 1 unemployed to 5 ≥ 7 years
    "installment_rate",       # 1 ≥ 35% to 4 < 20% of income
    "present_residence",      # 1 < 1 year to 4 ≥ 7 years
    "property",               # 1 none to 4 real estate
    "number_credits",         # 1 = one credit to 4 = ≥ 6 credits
    "job",                    # 1 unskilled to 4 highly qualified
]

nominal_categorical_cols = [
    "purpose",
    "personal_status_sex",
    "other_debtors",
    "other_installment_plans",
    "housing",
]

categorical_cols = (
    binary_categorical_cols
    + ordinal_categorical_cols
    + nominal_categorical_cols
)

print("Quantitative:", len(quantitative_cols))
print("Binary:      ", len(binary_categorical_cols))
print("Ordinal:     ", len(ordinal_categorical_cols))
print("Nominal:     ", len(nominal_categorical_cols))
print("Total predictors:", len(quantitative_cols) + len(categorical_cols))

Quantitative: 3
Binary:       3
Ordinal:      9
Nominal:      5
Total predictors: 20


#### EDA

In [8]:
def bad_rate_by_cat(df, col):
    tab = df.groupby(col)["bad"].agg(n="size", bad_rate="mean")
    tab = tab.sort_values("bad_rate")
    tab["spread"] = tab["bad_rate"].max() - tab["bad_rate"].min()
    tab["col"] = col
    return tab

def bad_rate_by_quant(df, col, q=5):
    binned = pd.qcut(df[col], q=q, duplicates="drop")
    tab = df.groupby(binned)["bad"].agg(n="size", bad_rate="mean")
    tab["spread"] = tab["bad_rate"].max() - tab["bad_rate"].min()
    tab["col"] = col
    return tab

# Categorical spread
cat_spreads = []
for col in categorical_cols:
    t = bad_rate_by_cat(train, col)
    if t["n"].min() >= 20:                 # skip variables with tiny cells
        cat_spreads.append({"variable": col, "spread": t["spread"].iloc[0]})

# Quantitative spread
quant_spreads = []
for col in quantitative_cols:
    t = bad_rate_by_quant(train, col)
    if t["n"].min() >= 20:
        quant_spreads.append({"variable": col, "spread": t["spread"].iloc[0]})

spreads = (
    pd.DataFrame(cat_spreads + quant_spreads)
    .sort_values("spread", ascending=False)
    .reset_index(drop=True)
)
display(spreads)


# Look at the top candidates in detail
for col in spreads["variable"].head(4):
    print(f"\n=== {col} ===")
    if col in categorical_cols:
        display(bad_rate_by_cat(train, col))
    else:
        display(bad_rate_by_quant(train, col))

,variable,spread
0,credit_history,0.507179
1,status,0.370364
2,other_debtors,0.292026
3,duration,0.291545
4,savings,0.240234
5,property,0.235559
6,foreign_worker,0.200893
7,amount,0.196584
8,employment_duration,0.180488
9,other_installment_plans,0.154999



=== credit_history ===


,n,bad_rate,spread,col
credit_history,,,,
4,199,0.135678,0.507179,credit_history
3,57,0.298246,0.507179,credit_history
2,378,0.335979,0.507179,credit_history
1,38,0.552632,0.507179,credit_history
0,28,0.642857,0.507179,credit_history



=== status ===


,n,bad_rate,spread,col
status,,,,
4,279,0.121864,0.370364,status
3,41,0.195122,0.370364,status
2,187,0.390374,0.370364,status
1,193,0.492228,0.370364,status



=== other_debtors ===


,n,bad_rate,spread,col
other_debtors,,,,
3,32,0.156250,0.292026,other_debtors
1,639,0.300469,0.292026,other_debtors
2,29,0.448276,0.292026,other_debtors



=== duration ===


,n,bad_rate,spread,col
duration,,,,
"(3.999, 12.0]",259,0.223938,0.291545,duration
"(12.0, 15.0]",50,0.180000,0.291545,duration
"(15.0, 24.0]",229,0.323144,0.291545,duration
"(24.0, 30.0]",39,0.282051,0.291545,duration
"(30.0, 72.0]",123,0.471545,0.291545,duration


**Notes :**
- Previous table shows how much the "bad creditor" rate moves across categories.
- larger spread, better separation between our target
- `credit_history` has the largest spread. Followed by `status`.
- `duration` is also high spread, but not monotone.



**How to grouping is done :**
- For the categorical variable we merge some categories into broader buckets that are easy to read and still meaningful for the bank.
- For the quantitative variable we bin the values into a few ranges.
- We then cross the two to get one group per (category bucket × bin) combination.

**Choosing the variables :**
- We pick one categorical and one ordinal so groupings shows both dims.
- Categorical: `credit_history` — previous repayment behaviour. Has the borrower defaulted before ?
- Quantitative: `duration` — credit duration in months, longer loans give more time for problems to appear.

**Grouping.**
- `credit_history` is ordinal (natural order in its categories). It has 4 categories :
- 0 (past delays) and 1 (critical account) → both mean previous problems.
- 2 (no credits / repaid duly) → ambiguous: either no history or repaid.
- 3 and 4 → both mean good history, paid duly.


 the third and fourthis collapsed into a small number of meaningful groups. Codes 3 and 4
  mean the customer has been paying existing credits duly, so they can be merged into
  one "reliable payer" group. Codes 0, 1 and 2 are kept separate because they mean very
  different things (delays, critical account, no credits).
- `duration` is binned into three groups: short (≤ 12 months), medium (13–24 months),
  long (> 24 months). This keeps the bins readable and gives enough observations per
  group.

The figure shows the bad-credit rate and the number of observations for each
(`credit_history` group × `duration` bin) combination.

In [ ]:
#

#### 4. Interpret the evidence

Identify the group with the highest bad-credit rate and the group containing the most bad
credits. Discuss the reliability of the pattern, considering group sizes and your grouping choices.
Give one useful takeaway for the bank.

---

In [ ]:
# Your data checks, modelling target and reproducible split.

In [ ]:
# Your exploration, joint groups, figure and supporting calculations.

## 2. Build and evaluate the model
Follow Question 2 on page 4. Select predictors, including at least one quantitative variable, define their levels and apply WOE. Fit logistic regression using only those WOE predictors; calculate IV and compare training/test ROC curves and AUC.

Justify choices and interpret results in report Part 2. Keep calculations, WOE tables and model coefficients here.


In [ ]:
# Your predictor selection, level definitions, WOE and IV calculations.

In [ ]:
# Your fitted model, interpretation support and ROC evaluation.

## 3. Construct the scorecard
Follow Question 3 on page 5. Use 600 points at good:bad odds of 50:1 and 20 points to double those odds. Verify the scorecard against model probabilities on all test rows and check its scale.

Put the full scorecard and test-score boxplots in report Part 3, highlighting the two cases requested. Keep derivations and checks here.


In [ ]:
# Your complete scorecard and verification on the test sample.

In [ ]:
# Your test-score boxplots and the two highlighted cases.

**Submission:** run all cells in order, check results and save with outputs. Email the notebook and report PDF to **andreas.goethals@kuleuven.be** by **10 November 2026, 23:59 Belgian time**.

Subject: `DSFB_A1_r1_r2`. Files: `A1_r1_r2.pdf` and `A1_r1_r2.ipynb`. Use your actual r-numbers, smaller first. Follow all guidelines on page 1.
